# Projet 2 : fidélisation client d'un site e-commerce
**Auteure : Guifo Maeva** · Python (Pandas, Scikit-learn, Matplotlib)

**Contexte :** un site e-commerce perd des clients. La direction veut savoir **qui sont ses meilleurs clients, qui risque de partir, et où le site perd des ventes**.

**Les 3 analyses du projet :**
1. **Segmentation RFM** : classer les clients selon leur valeur.
2. **Modèle prédictif de churn** : prédire quels clients vont arrêter d'acheter.
3. **Tunnel de conversion** : trouver l'étape où les visiteurs abandonnent.

**Comment utiliser ce notebook :** exécute les cellules une par une (Maj + Entrée) et lis le texte avant chacune.

## Étape 1 : charger les données
Sur **Google Colab**, une fenêtre te demande de choisir `Donnees_Brutes_Fidelisation.xlsx`. Le fichier contient 3 onglets :
- **Clients** : 3 000 clients (ville, âge, canal d'acquisition…) ;
- **Commandes** : toutes les commandes de 2024 et 2025 ;
- **Sessions** : 30 000 visites du site d'octobre à décembre 2025, avec l'étape la plus loin atteinte.

In [ ]:
try:
    from google.colab import files
    files.upload()
except ImportError:
    pass
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
FICHIER = "Donnees_Brutes_Fidelisation.xlsx"
onglets = pd.read_excel(FICHIER, sheet_name=None)
clients = onglets["Clients"]; commandes = onglets["Commandes"]; sessions = onglets["Sessions"]
print("Lignes :", len(clients), len(commandes), len(sessions))

## Étape 2 : explorer les données
On cherche les problèmes avant de nettoyer : cases vides, villes écrites de façons différentes, doublons, commandes annulées.

In [ ]:
print(clients.isna().sum()); print(clients["ville"].value_counts().head(12))
print("Doublons commandes :", commandes.duplicated().sum()); print(commandes["statut"].value_counts())

**Ce qu'on observe :**
- 60 villes vides et 25 âges vides ;
- des villes en majuscules (« LILLE » et « Lille » sont comptées séparément) ;
- 40 commandes en double ;
- des commandes **annulées**, à exclure du chiffre d'affaires ;
- dans l'onglet Commandes, le montant est écrit en texte (« 54,20 € ») et la date au format 31/12/2025.

## Étape 3 : nettoyer les clients
- Villes : suppression des espaces et même écriture pour toutes (« LILLE » devient « Lille ») ; villes vides remplacées par « Inconnue ».
- Âges vides : remplacés par la **médiane**, qui n'est pas influencée par les valeurs extrêmes.

In [ ]:
c = clients.copy()
c["ville"] = c["ville"].str.strip().str.title().fillna("Inconnue")
c["age"] = c["age"].fillna(c["age"].median()).astype(int)
c["date_inscription"] = pd.to_datetime(c["date_inscription"])
print(c["ville"].value_counts())

## Étape 4 : nettoyer les commandes
Suppression des doublons, conversion du montant en nombre, conversion des dates, puis on garde **uniquement les commandes livrées**.

In [ ]:
o = commandes.drop_duplicates().copy()
o["montant"] = o["montant"].str.replace("€","").str.replace(",",".").str.strip().astype(float)
o["date_commande"] = pd.to_datetime(o["date_commande"], format="%d/%m/%Y")
o["retournee"] = o["retournee"] == "Oui"; o["code_promo"] = o["code_promo"] == "Oui"
o = o[o["statut"] == "Livrée"]
print("Commandes livrées :", len(o), "| CA total :", round(o.montant.sum()))

## Étape 5 : la segmentation RFM
Pour chaque client, on calcule 3 indicateurs :
- **R (Récence)** : nombre de jours depuis la dernière commande. Plus c'est petit, mieux c'est.
- **F (Fréquence)** : nombre de commandes.
- **M (Montant)** : chiffre d'affaires total du client.

Chaque indicateur reçoit une **note de 1 à 5** (5 = le meilleur), en découpant les clients en 5 groupes égaux (quintiles). Les notes R et F permettent ensuite de classer chaque client dans un **segment** :

| Segment | Règle | Signification |
|---|---|---|
| Champions | R ≥ 4 et F ≥ 4 | Achètent souvent et récemment |
| Fidèles | R ≥ 3 et F ≥ 3 | Bons clients réguliers |
| Nouveaux | R ≥ 4 et F ≤ 2 | Premiers achats récents |
| À risque | R ≤ 2 et F ≥ 3 | Bons clients qui n'achètent plus |
| Perdus | R ≤ 2 et F ≤ 2 | Peu d'achats, il y a longtemps |
| À surveiller | Les autres | Profils intermédiaires |

In [ ]:
DATE_ANALYSE = pd.Timestamp("2026-01-01")
rfm = o.groupby("id_client").agg(derniere=("date_commande","max"), frequence=("id_commande","count"), montant=("montant","sum"))
rfm["recence"] = (DATE_ANALYSE - rfm["derniere"]).dt.days
rfm["R"] = pd.qcut(rfm["recence"].rank(method="first"), 5, labels=[5,4,3,2,1]).astype(int)
rfm["F"] = pd.qcut(rfm["frequence"].rank(method="first"), 5, labels=[1,2,3,4,5]).astype(int)
rfm["M"] = pd.qcut(rfm["montant"].rank(method="first"), 5, labels=[1,2,3,4,5]).astype(int)
def segment(r):
    if r.R >= 4 and r.F >= 4: return "Champions"
    if r.R >= 3 and r.F >= 3: return "Fidèles"
    if r.R >= 4 and r.F <= 2: return "Nouveaux"
    if r.R <= 2 and r.F >= 3: return "À risque"
    if r.R <= 2 and r.F <= 2: return "Perdus"
    return "À surveiller"
rfm["segment"] = rfm.apply(segment, axis=1)
seg = rfm.groupby("segment").agg(clients=("frequence","size"), recence_moy=("recence","mean"),
        frequence_moy=("frequence","mean"), ca=("montant","sum"))
seg["part_clients_%"] = seg.clients/seg.clients.sum()*100; seg["part_ca_%"] = seg.ca/seg.ca.sum()*100
seg["ca_par_client"] = seg.ca/seg.clients
print(seg.round(1).sort_values("ca", ascending=False))

**Lecture :** les **Champions** représentent environ un quart des clients mais **la moitié du chiffre d'affaires**. Le segment **À risque** est le plus préoccupant : 483 anciens bons clients (environ 590 € de CA chacun, soit 18 % du CA total) qui n'ont plus acheté depuis plus d'un an en moyenne.

## Étape 6 : préparer le modèle de churn
**Churn** = un client qui arrête d'acheter.

Pour entraîner un modèle, il faut des exemples dont on connaît la réponse. On fait donc comme si on était le **1er octobre 2025** :
- les **variables** (récence, fréquence, montant, taux de retour, part des commandes avec code promo…) sont calculées avec les commandes **avant** le 1er octobre ;
- la **réponse** (churn = 1) est : le client n'a **rien acheté entre octobre et décembre 2025**.

Ainsi, le modèle apprend à prédire l'avenir à partir du passé, sans « tricher » avec des informations futures.

In [ ]:
COUPURE = pd.Timestamp("2025-10-01")
avant = o[o.date_commande < COUPURE]; apres = o[o.date_commande >= COUPURE]
X = avant.groupby("id_client").agg(derniere=("date_commande","max"), premiere=("date_commande","min"),
        frequence=("id_commande","count"), montant=("montant","sum"),
        taux_retour=("retournee","mean"), part_promo=("code_promo","mean"))
X["recence"] = (COUPURE - X.derniere).dt.days
X["anciennete"] = (COUPURE - X.premiere).dt.days
X["panier_moyen"] = X.montant / X.frequence
X["jours_entre_commandes"] = X.anciennete / X.frequence
X = X.join(c.set_index("id_client")[["age","canal_acquisition"]])
X["churn"] = (~X.index.isin(apres.id_client)).astype(int)
print("Clients :", len(X), "| taux de churn :", round(X.churn.mean()*100,1), "%")
print(X.groupby("canal_acquisition").churn.mean().round(3))

**À noter :** les clients venus par un **code promo influenceur** partent beaucoup plus (82 %) que les autres (54 % à 61 %). Ce sont souvent des clients attirés par la réduction, pas par la marque.

## Étape 7 : entraîner et évaluer les modèles
On sépare les clients en deux : **75 % pour apprendre** (train) et **25 % pour tester** (test), afin d'évaluer le modèle sur des clients qu'il n'a jamais vus.

On compare deux modèles :
- la **régression logistique**, simple et facile à expliquer ;
- la **forêt aléatoire** (Random Forest), qui combine de nombreux arbres de décision.

**Les indicateurs :**
- **Accuracy** : part des prédictions correctes ;
- **AUC** : capacité à distinguer les clients qui partent de ceux qui restent (0,5 = hasard, 1 = parfait) ;
- **Précision** : parmi les clients prédits « churn », combien partent vraiment ;
- **Rappel** : parmi les clients qui partent vraiment, combien le modèle en détecte.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score, precision_score, confusion_matrix
VARS = ["recence","frequence","montant","panier_moyen","anciennete","jours_entre_commandes","taux_retour","part_promo","age"]
Xm = pd.get_dummies(X[VARS + ["canal_acquisition"]], columns=["canal_acquisition"], dtype=int)
y = X["churn"]
X_train, X_test, y_train, y_test = train_test_split(Xm, y, test_size=0.25, random_state=42, stratify=y)
modeles = {"Régression logistique": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
           "Forêt aléatoire": RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42)}
for nom, m in modeles.items():
    m.fit(X_train, y_train); p = m.predict_proba(X_test)[:,1]; pr = (p>=0.5).astype(int)
    print(f"{nom:22s} accuracy {accuracy_score(y_test,pr):.2f} | AUC {roc_auc_score(y_test,p):.2f} | précision {precision_score(y_test,pr):.2f} | rappel {recall_score(y_test,pr):.2f}")
rf = modeles["Forêt aléatoire"]
print(confusion_matrix(y_test, rf.predict(X_test)))
imp = pd.Series(rf.feature_importances_, index=Xm.columns).sort_values(ascending=False)
print(imp.round(3).head(10))

**Lecture :** les deux modèles ont une **AUC de 0,91**, ce qui est très bon. La forêt aléatoire détecte **88 % des clients qui partent** (rappel). La matrice de confusion le confirme : sur 445 clients partis du jeu de test, 391 ont été détectés.

**Variables les plus importantes :** la **récence** arrive très largement en tête, suivie du **délai moyen entre deux commandes**. Autrement dit, un client qui commence à espacer ses achats est un signal d'alerte.

## Étape 8 : appliquer le modèle aux clients actuels
On calcule maintenant les mêmes variables au 1er janvier 2026 et on donne à chaque client une **probabilité de churn**. C'est ce score que l'équipe marketing peut utiliser pour cibler ses actions.

In [ ]:
Z = o.groupby("id_client").agg(derniere=("date_commande","max"), premiere=("date_commande","min"),
        frequence=("id_commande","count"), montant=("montant","sum"),
        taux_retour=("retournee","mean"), part_promo=("code_promo","mean"))
Z["recence"] = (DATE_ANALYSE - Z.derniere).dt.days; Z["anciennete"] = (DATE_ANALYSE - Z.premiere).dt.days
Z["panier_moyen"] = Z.montant/Z.frequence; Z["jours_entre_commandes"] = Z.anciennete/Z.frequence
Z = Z.join(c.set_index("id_client")[["age","canal_acquisition"]])
Zm = pd.get_dummies(Z[VARS+["canal_acquisition"]], columns=["canal_acquisition"], dtype=int).reindex(columns=Xm.columns, fill_value=0)
rfm["proba_churn"] = rf.predict_proba(Zm)[:,1]
cibles = rfm[(rfm.segment.isin(["À risque","Fidèles","Champions"])) & (rfm.proba_churn>=0.5)]
print("Clients de valeur à risque :", len(cibles), "| CA historique :", round(cibles.montant.sum()))
print(cibles.segment.value_counts())

## Étape 9 : le tunnel de conversion
Pour chaque appareil, on compte combien de visites atteignent chaque étape (visite, fiche produit, ajout panier, début du paiement, achat), puis on calcule le **taux de passage** d'une étape à la suivante.

In [ ]:
ETAPES = ["Visite","Fiche produit","Ajout panier","Début paiement","Achat"]
s = sessions.copy(); s["niveau"] = s.etape_max_atteinte.map({e:i for i,e in enumerate(ETAPES)})
fun = pd.DataFrame({e: s.groupby("appareil").niveau.apply(lambda n: (n>=i).sum()) for i,e in enumerate(ETAPES)})
print(fun)
taux = pd.DataFrame({f"{ETAPES[i-1]} → {ETAPES[i]}": fun[ETAPES[i]]/fun[ETAPES[i-1]]*100 for i in range(1,5)}).round(1)
taux["Conversion globale %"] = (fun["Achat"]/fun["Visite"]*100).round(2)
print(taux.T)

**Lecture : le point de friction est le paiement sur mobile.**
- Sur ordinateur, **77 %** des visiteurs qui commencent à payer finalisent leur achat.
- Sur mobile, seulement **42 %**. Plus de la moitié abandonne pendant le paiement.
- Or le mobile représente **62 % des visites**. La conversion globale sur mobile (3 %) est trois fois plus faible que sur ordinateur (10 %).

## Étape 10 : estimer l'impact des actions
Hypothèse prudente : si le taux de finalisation du paiement sur mobile passe de 42 % à **65 %** (sans atteindre le niveau de l'ordinateur), combien de ventes en plus ?

In [ ]:
pm = o.montant.mean()
mob = fun.loc["Mobile"]; t_des = fun.loc["Ordinateur","Achat"]/fun.loc["Ordinateur","Début paiement"]
cible_mob = 0.65
gain_achats = mob["Début paiement"]*cible_mob - mob["Achat"]
print(f"panier moyen {pm:.2f} | taux paiement mobile {mob['Achat']/mob['Début paiement']:.2%} vs ordi {t_des:.2%}")
print(f"achats supplémentaires sur 3 mois (mobile à 65%) : {gain_achats:.0f} → CA {gain_achats*pm:,.0f} € / trimestre, {gain_achats*pm*4:,.0f} € / an")
ar = rfm[rfm.segment=="À risque"]
print(f"À risque : {len(ar)} clients, panier moyen {ar.montant.sum()/ar.frequence.sum():.2f}, freq moy {ar.frequence.mean():.1f}")

## Étape 11 : graphiques

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
seg.sort_values("ca")["part_ca_%"].plot.barh(ax=axes[0], color="#16324F")
axes[0].set_title("Part du CA par segment (%)")
(taux["Début paiement → Achat"]).plot.bar(ax=axes[1], color=["#E24B4A", "#16324F", "#16324F"], rot=0)
axes[1].set_title("Taux de finalisation du paiement (%)")
imp.head(6).sort_values().plot.barh(ax=axes[2], color="#E8A33D")
axes[2].set_title("Variables les plus importantes (churn)")
plt.tight_layout(); plt.show()

## Étape 12 : exporter les résultats (pour Power BI ou Excel)

In [ ]:
export = rfm.join(c.set_index("id_client")[["ville", "age", "canal_acquisition"]])
with pd.ExcelWriter("resultats_fidelisation.xlsx") as w:
    export.reset_index().to_excel(w, sheet_name="clients_segments", index=False)
    seg.round(2).to_excel(w, sheet_name="resume_segments")
    fun.to_excel(w, sheet_name="tunnel_volumes"); taux.to_excel(w, sheet_name="tunnel_taux")
    imp.round(4).to_frame("importance").to_excel(w, sheet_name="importance_variables")
try:
    files.download("resultats_fidelisation.xlsx")
except NameError:
    print("Fichier enregistré dans le dossier du notebook")

## Conclusion et recommandations

**1. Corriger le paiement sur mobile (priorité n°1).**
Le mobile représente 62 % des visites, mais seulement 42 % des paiements commencés y sont finalisés, contre 77 % sur ordinateur. Actions : paiement express (Apple Pay, Google Pay, PayPal), achat sans création de compte, formulaire simplifié et testé sur petit écran. **Impact estimé : environ +19 000 € de CA par trimestre, soit environ +77 000 € par an**, si le taux atteint 65 %.

**2. Réactiver les clients « À risque ».**
483 anciens bons clients (environ 9,6 commandes chacun, 18 % du CA historique) n'achètent plus. Actions : campagne de réactivation personnalisée (email avec recommandations basées sur leurs achats passés, offre de retour limitée dans le temps), en priorisant les clients au plus fort montant historique.

**3. Utiliser le score de churn pour agir avant le départ.**
Le modèle (AUC 0,91) montre que l'espacement des commandes est le premier signal d'alerte. Actions : calculer le score chaque mois et déclencher un email automatique quand un client Fidèle ou Champion dépasse 50 % de probabilité de churn.

**4. Revoir les partenariats influenceurs.**
82 % des clients venus par code promo influenceur ne rachètent pas, contre 54 % à 61 % pour les autres canaux. Actions : mesurer la valeur à long terme de ces clients avant de renouveler les partenariats, et accompagner leur 1er achat d'un parcours de fidélisation.

**5. Protéger les Champions.**
Un quart des clients génère la moitié du CA. Un programme de fidélité (accès anticipé, livraison offerte) sécurise cette base.

**Limites :** les données sont simulées ; le seuil de churn (aucun achat en 3 mois) dépend du rythme d'achat du secteur ; l'impact estimé doit être confirmé par un test A/B avant de généraliser.